In [0]:
from pyspark.sql.functions import (
    col, lit, concat, when, current_date,
    year, month, dayofmonth, dayofweek, weekofyear, quarter,
    date_format, explode, sequence, to_date,
    sum as spark_sum, max as spark_max,
    count as spark_count, coalesce,
    regexp_replace, upper, trim
)
from pyspark.sql import Row

def to_num(c):
    return when(
        regexp_replace(col(c), "[^0-9.]", "") == "", lit(None)
    ).otherwise(
        regexp_replace(col(c), "[^0-9.]", "").cast("double")
    )

In [0]:
silver_site = spark.table("cre_catalog.silver.dim_site")
silver_bldg = spark.table("cre_catalog.silver.dim_building")
silver_tax = spark.table("cre_catalog.silver.fact_tax_assessment")
silver_sale = spark.table("cre_catalog.silver.fact_sale_event")

In [0]:
# ══════════════════════════════════════════════════════════
# 1. dim_property
#    Grain: one row per site_id (current snapshot)
#    Joins dim_site + aggregated building metrics
# ══════════════════════════════════════════════════════════

bldg_agg = (
    silver_bldg
    .filter(col("is_current") == True)
    .groupBy("site_id")
    .agg(
        spark_count("building_id").alias("num_buildings"),
        spark_sum("gross_sqft").alias("total_gross_sqft"),
        spark_sum("num_units").alias("total_units"),
        spark_max("year_built").alias("newest_year_built"),
        spark_sum("office_sqft").alias("total_office_sqft"),
        spark_sum("retail_sqft").alias("total_retail_sqft"),
        spark_sum("residential_sqft").alias("total_residential_sqft"),
        spark_sum("garage_sqft").alias("total_garage_sqft")
    )
)

dim_property = (
    silver_site
    .filter(col("is_current") == True)
    .join(bldg_agg, "site_id", "left")
    .select(
        silver_site["site_id"].alias("property_id"),
        silver_site["parid"],
        silver_site["boro"],
        silver_site["block"],
        silver_site["lot"],
        silver_site["zip_code"],
        silver_site["zoning"],
        silver_site["street_name"],
        silver_site["house_num_lo"],
        silver_site["house_num_hi"],
        silver_site["latitude"],
        silver_site["longitude"],
        silver_site["h3_index"],
        silver_site["polygon_wkt"],
        silver_site["lot_area"],
        silver_site["lot_front"],
        silver_site["lot_depth"],
        silver_site["land_area"],
        silver_site["corner"],
        silver_site["lot_irreg"],
        silver_site["parent_site_id"],
        bldg_agg["num_buildings"],
        bldg_agg["total_gross_sqft"],
        bldg_agg["total_units"],
        bldg_agg["newest_year_built"],
        bldg_agg["total_office_sqft"],
        bldg_agg["total_retail_sqft"],
        bldg_agg["total_residential_sqft"],
        bldg_agg["total_garage_sqft"],
        silver_site["effective_from"],
        silver_site["effective_to"],
        silver_site["is_current"]
    )
)

In [0]:
dim_property.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.gold.dim_property")

print(f"✅ dim_property rows: {dim_property.count():,}")

In [0]:
# ══════════════════════════════════════════════════════════
# 5. dim_building
#    Promoted directly from silver — already clean
# ══════════════════════════════════════════════════════════
(
    spark.table("cre_catalog.silver.dim_building")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cre_catalog.gold.dim_building")
)
print(f"✅ dim_building promoted to gold")

In [0]:
# ══════════════════════════════════════════════════════════
# 6. dim_floor
#    Promoted directly from silver — already clean
# ══════════════════════════════════════════════════════════
(
    spark.table("cre_catalog.silver.dim_floor")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cre_catalog.gold.dim_floor")
)
print(f"✅ dim_floor promoted to gold")

In [0]:
# ══════════════════════════════════════════════════════════
# 2. dim_owner
#    Promoted directly from silver — already clean
# ══════════════════════════════════════════════════════════
(
    spark.table("cre_catalog.silver.dim_owner")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cre_catalog.gold.dim_owner")
)
print("✅ dim_owner promoted to gold")

In [0]:

date_spine = (
    spark.range(1)
    .select(
        explode(
            sequence(
                to_date(lit("2000-01-01")),
                to_date(lit("2035-12-31")),
                expr("interval 1 day")
            )
        ).alias("full_date")
    )
)

dim_date = (
    date_spine
    .withColumn("date_id",       date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year",           year("full_date"))
    .withColumn("quarter",        quarter("full_date"))
    .withColumn("month",          month("full_date"))
    .withColumn("month_name",     date_format("full_date", "MMMM"))
    .withColumn("day_of_month",   dayofmonth("full_date"))
    .withColumn("day_of_week",    dayofweek("full_date"))       # 1=Sun
    .withColumn("day_name",       date_format("full_date", "EEEE"))
    .withColumn("is_weekend",     when(dayofweek("full_date").isin(1,7), True).otherwise(False))
    .withColumn("week_of_year",   weekofyear("full_date"))
    .withColumn("fiscal_year",    when(month("full_date") >= 7, year("full_date") + 1)
                                   .otherwise(year("full_date")))  # Jul-Jun fiscal
    .withColumn("yyyymm",         date_format("full_date", "yyyyMM").cast("int"))
    .select(
        "date_id", "full_date", "year", "quarter", "month", "month_name",
        "day_of_month", "day_of_week", "day_name", "is_weekend",
        "week_of_year", "fiscal_year", "yyyymm"
    )
)

In [0]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.gold.dim_date")

print(f"✅ dim_date rows: {dim_date.count():,}")

In [0]:
# ══════════════════════════════════════════════════════════
# 4. dim_event_type
#    Static lookup — covers all event categories
# ══════════════════════════════════════════════════════════

event_types = [
    Row(event_type_id=1,  event_category="TAX",   event_type_name="Annual Assessment",   description="Yearly tax assessment snapshot"),
    Row(event_type_id=2,  event_category="TAX",   event_type_name="Supplemental Filing", description="Mid-year or corrected filing"),
    Row(event_type_id=3,  event_category="SALE",  event_type_name="Arms Length Sale",    description="Market-rate ownership transfer"),
    Row(event_type_id=4,  event_category="SALE",  event_type_name="Non-Arms Length",     description="Transfer between related parties"),
    Row(event_type_id=5,  event_category="SALE",  event_type_name="Foreclosure Sale",    description="Distressed or bank-owned transfer"),
]

spark.createDataFrame(event_types) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.gold.dim_event_type")

print("✅ dim_event_type written")

In [0]:

fact_tax = (
    silver_tax
    .withColumn(
        # date_id = Jan 1 of tax year → links to dim_date
        "date_id",
        concat(col("tax_year").cast("string"), lit("0101")).cast("int")
    )
    .withColumn("event_type_id", lit(1))  # 1 = Annual Assessment
    .select(
        "assessment_id",            # PK
        "site_id",                  # FK → dim_property
        "owner_id",                 # FK → dim_owner
        "date_id",                  # FK → dim_date
        "event_type_id",            # FK → dim_event_type
        "tax_year",
        "period",
        # Classification
        "tax_class",
        "tax_class_prior_year",
        "tax_class_tentative",
        "tax_class_final",
        "bldg_class",
        "zoning",
        "roll_section",
        # Prior year snapshot
        "py_mkt_land", "py_mkt_total",
        "py_act_land", "py_act_total",
        "py_trn_land", "py_trn_total",
        "py_txb_total",
        # Tentative snapshot
        "ten_mkt_land", "ten_mkt_total",
        "ten_act_land", "ten_act_total",
        "ten_trn_land", "ten_trn_total",
        "ten_txb_total",
        # Combined snapshot
        "cbn_mkt_land", "cbn_mkt_total",
        "cbn_act_land", "cbn_act_total",
        "cbn_trn_land", "cbn_trn_total",
        "cbn_txb_total",
        # Final snapshot
        "fin_mkt_land", "fin_mkt_total",
        "fin_act_land", "fin_act_total",
        "fin_trn_land", "fin_trn_total",
        "fin_txb_total",
        # Current snapshot
        "cur_mkt_land", "cur_mkt_total",
        "cur_act_land", "cur_act_total",
        "cur_trn_land", "cur_trn_total",
        "cur_txb_total",
        # Building metrics at time of filing
        "gross_sqft", "residential_sqft", "office_sqft",
        "retail_sqft", "hotel_sqft", "loft_sqft",
        "factory_sqft", "warehouse_sqft", "storage_sqft",
        "garage_sqft", "other_sqft",
        "num_stories", "num_bldgs", "num_units", "year_built",
        # Flags
        "building_in_progress", "new_drop_flag",
        "py_tax_flag", "ten_tax_flag", "fin_tax_flag", "cur_tax_flag",
        "load_date"
    )
)

# ── Sanity check ──────────────────────────────────────────
print(f"fact_tax_assessment rows   : {fact_tax.count():,}")
print(f"Distinct site_ids          : {fact_tax.select('site_id').distinct().count():,}")
print(f"Distinct tax years         : {fact_tax.select('tax_year').distinct().count():,}")

fact_tax.groupBy("tax_year").count().orderBy("tax_year").show(30)


In [0]:
fact_tax.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.gold.fact_tax_assessment")

print("✅ fact_tax_assessment written to Gold")

In [0]:
# ══════════════════════════════════════════════════════════
# 2. fact_sale
#    Promoted from silver.fact_sale_event + date_id added
#    Grain: one row per inferred ownership transfer
# ══════════════════════════════════════════════════════════

fact_sale = (
    silver_sale
    .withColumn(
        "date_id",
        concat(col("sale_year").cast("string"), lit("0101")).cast("int")
    )
    .withColumn("event_type_id", lit(3))  # 3 = Arms Length Sale (default)
    .select(
        col("sale_event_id"),                       # PK
        col("site_id").alias("property_id"),        # FK → dim_property
        col("seller_id").alias("seller_owner_id"),  # FK → dim_owner
        col("buyer_id").alias("buyer_owner_id"),    # FK → dim_owner
        col("date_id"),                             # FK → dim_date
        col("event_type_id"),                       # FK → dim_event_type
        col("sale_year"),
        col("prior_ownership_year"),
        col("seller_name"),
        col("buyer_name"),
        col("tax_class"),
        col("bldg_class"),
        col("zoning"),

        col("source_type"),                         # INFERRED
        col("load_date")
    )
)

# ── Sanity check ──────────────────────────────────────────
print(f"fact_sale rows             : {fact_sale.count():,}")
print(f"Distinct properties        : {fact_sale.select('property_id').distinct().count():,}")

fact_sale.groupBy("sale_year").count().orderBy("sale_year").show(30)


In [0]:
fact_sale.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.gold.fact_sale")

print("✅ fact_sale written to Gold")

In [0]:
# ══════════════════════════════════════════════════════════
#    dim_hotel
#    Promoted directly from silver
# ══════════════════════════════════════════════════════════
(
    spark.table("cre_catalog.silver.dim_hotel")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cre_catalog.gold.dim_hotel")
)
print(f"✅ dim_hotel promoted to gold: {spark.table('cre_catalog.gold.dim_hotel').count():,} rows")

In [0]:
# ══════════════════════════════════════════════════════════
#    dim_restaurant
#    Promoted directly from silver
# ══════════════════════════════════════════════════════════
(
    spark.table("cre_catalog.silver.dim_restaurant")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cre_catalog.gold.dim_restaurant")
)
print(f"✅ dim_restaurant promoted to gold: {spark.table('cre_catalog.gold.dim_restaurant').count():,} rows")

In [0]:
# ══════════════════════════════════════════════════════════
#    fact_site_amenity
#    Promoted directly from silver
#    Grain: one row per site + amenity pair within 1km
# ══════════════════════════════════════════════════════════
spark.table("cre_catalog.silver.fact_site_amenity") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.gold.fact_site_amenity")
    
print(f"✅ fact_site_amenity written to Gold: {spark.table('cre_catalog.gold.fact_site_amenity').count():,} rows")